In [ ]:
import requests, threading, queue, time, json, os, logging
import boto3
import pandas as pd
import io

BASE = "https://openlibrary.org"
NUM_THREADS = 8
AUTHOR_THREADS = 4
SLEEP = 0.1
RETRY = 3

work_queue = queue.Queue(maxsize=10000)
author_queue = queue.Queue(maxsize=10000)

lock = threading.Lock()
session = requests.Session()

logging.basicConfig(level=logging.INFO, filename="crawler.log")

s3 = boto3.client("s3")
bucket = 'mhai-bk'

# ================= GLOBAL DEDUP =================
seen_editions = set()
seen_authors = set()

def clean_dataframe(df):
    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
        )
    return df
# ================= SAVE PARQUET =================
def upload_parquet(records, prefix):
    if not records:
        return

    df = pd.DataFrame(records)

    df = clean_dataframe(df)
    #  ghi parquet vào RAM (không cần file local)
    buffer = io.BytesIO()
    df.to_parquet(buffer, engine="pyarrow", compression="snappy", index=False)

    buffer.seek(0)

    #  tạo key
    key = f"raw_data/{prefix}/part-{int(time.time()*1000)}.parquet"

    #  upload lên S3
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=buffer.getvalue(),
        ContentType="application/octet-stream"
    )

# ================= FETCH WITH RETRY =================
def fetch(url):
    for i in range(RETRY):
        try:
            r = session.get(url, timeout=10)
            if r.status_code == 200:
                return r.json()
        except Exception:
            logging.warning(f"Retry {i} failed: {url}")
            time.sleep(1)
    return None

# ================= CRAWL WORKS =================
def crawl_subject(subject):
    offset = 0
    seen_local = set()

    while True:
        url = f"{BASE}/subjects/{subject}.json?limit=50&offset={offset}"
        data = fetch(url)

        if not data or not data.get("works"):
            break

        records = []

        for w in data["works"]:
            wid = w["key"]

            if wid in seen_local:
                continue
            seen_local.add(wid)

            records.append(w)

            # push vào queue
            work_queue.put(wid)

            for a in w.get("authors", []):
                aid = a.get("author", {}).get("key")
                if aid:
                    author_queue.put(aid)

        print(f"[WORKS] {subject} -> {len(records)}")

        if records:
            with lock:
                upload_parquet(records, "works")
            

        offset += 50
        time.sleep(SLEEP)

# ================= EDITIONS WORKER =================
def worker_editions():
    while True:
        try:
            work_id = work_queue.get(timeout=5)
        except:
            continue  #  không break

        offset = 0
        batch = []

        while True:
            url = f"{BASE}{work_id}/editions.json?limit=100&offset={offset}"
            data = fetch(url)

            if not data or not data.get("entries"):
                break

            for e in data["entries"]:
                key = e.get("key")

                with lock :
                    if key and key not in seen_editions:
                        seen_editions.add(key)
                        batch.append(e)

                #  THÊM ĐOẠN NÀY
                for a in e.get("authors", []):
                    aid = a.get("key")
                    if aid:
                        author_queue.put(aid)
            offset += 100

        if batch:
            print(f"[EDITIONS] {work_id} -> {len(batch)}")
            with lock:
                upload_parquet(batch, "editions")

        work_queue.task_done()
        time.sleep(SLEEP)
        print(f"author_queue size: {author_queue.qsize()}")

# ================= AUTHORS WORKER =================
def worker_authors():
    batch_authors = []
    while True:
        try:
            aid = author_queue.get(timeout=5)
        except:
            continue  #  không break

        if aid in seen_authors:
            author_queue.task_done()
            continue

        seen_authors.add(aid)
        data = fetch(f"{BASE}{aid}.json")

        if data:
            print(f"[AUTHOR] {aid}")
            batch_authors.append(data)
        if batch_authors:
            with lock:
                upload_parquet(batch_authors, "authors")

        author_queue.task_done()
        time.sleep(SLEEP)
        print("Author worker alive")

# ================= MAIN =================
def main():
    subjects = [
        "philosophy","psychology","education","medicine","law",
        "engineering","mathematics","physics","chemistry","astronomy",

        "geography","politics","economics","sociology","anthropology",
        "religion","mythology","literature","poetry","drama",

        "comics","graphic_novels","children","young_adult","horror",
        "thriller","mystery","crime","adventure","travel",

        "cooking","food","health","fitness","self_help",
        "personal_development","parenting","relationships","spirituality",

        "design","architecture","photography","fashion","film",
        "television","media","journalism","communication",

        "programming","computer_science","data_science","artificial_intelligence",
        "machine_learning","cybersecurity","software_engineering","cloud_computing",

        "finance","investing","marketing","management","entrepreneurship",
        "startups","leadership","strategy","sales",

        "environment","ecology","sustainability","climate_change","agriculture",
        "nature","wildlife","marine_biology","earth_science",

        "sports","games","chess","board_games","video_games",
        "esports","olympics","football","basketball",

        "transportation","aviation","railways","automobiles","space",
        "rockets","nasa","exploration",

        "languages","linguistics","translation","grammar","writing",
        "publishing","editing","storytelling",

        "culture","traditions","folklore","heritage","civilization",
        "ancient_history","modern_history","world_history",

        "ethics","logic","metaphysics","epistemology","aesthetics",

        "statistics","probability","algebra","geometry","calculus",

        "quantum_physics","thermodynamics","optics","nuclear_physics",

        "organic_chemistry","inorganic_chemistry","biochemistry",

        "genetics","microbiology","neuroscience","zoology","botany",

        "public_health","epidemiology","nutrition","pharmacology",

        "legal_studies","criminal_law","international_law",

        "urban_planning","construction","real_estate",

        "supply_chain","logistics","operations","manufacturing",

        "human_resources","organizational_behavior",

        "ethnography","demography","social_theory",

        "comparative_religion","theology",

        "creative_writing","screenwriting","playwriting",

        "animation","visual_effects","game_design",

        "ux_ui","product_design","industrial_design",

        "blockchain","cryptocurrency","fintech",

        "big_data","data_engineering","data_analytics",

        "deep_learning","nlp","computer_vision",

        "iot","embedded_systems","robotics",

        "renewable_energy","solar_energy","wind_energy",

        "mining","petroleum","materials_science",

        "military","war","defense","strategy",

        "genealogy","archives","libraries",

        "tourism","hospitality","event_management"
    ]

    print("Start workers...")

    #  start workers (daemon)
    for _ in range(NUM_THREADS):
        threading.Thread(target=worker_editions, daemon=True).start()

    for _ in range(AUTHOR_THREADS):
        threading.Thread(target=worker_authors, daemon=True).start()

    #  start works threads
    work_threads = []
    for s in subjects:
        t = threading.Thread(target=crawl_subject, args=(s,))
        t.start()
        work_threads.append(t)

    #  chờ works xong
    for t in work_threads:
        t.join()

    #  chờ queue xử lý xong
    work_queue.join()
    author_queue.join()

    print("DONE")

if __name__ == "__main__":
    main()

Start workers...
[WORKS] technology -> 50
[WORKS] art -> 50
[WORKS] biography -> 50
[WORKS] history -> 50
[WORKS] music -> 50
[WORKS] science_fiction -> 50
[WORKS] fantasy -> 50
[WORKS] science -> 50
[WORKS] business -> 50
[WORKS] romance -> 50
Author worker alive
[EDITIONS] /works/OL244537W -> 100
[WORKS] biography -> 50
author_queue size: 253
[WORKS] art -> 50
Author worker alive
[EDITIONS] /works/OL14903313W -> 65
author_queue size: 354
[EDITIONS] /works/OL88800W -> 100
author_queue size: 660
Author worker alive
[EDITIONS] /works/OL695408W -> 89
author_queue size: 710
author_queue size: 710
[EDITIONS] /works/OL46884W -> 64
author_queue size: 959
author_queue size: 959
[EDITIONS] /works/OL14903327W -> 92
author_queue size: 959
[WORKS] technology -> 48
author_queue size: 959
[WORKS] science_fiction -> 50
[WORKS] music -> 50
[WORKS] history -> 50
[WORKS] fantasy -> 50
[AUTHOR] /authors/OL2815944A
Author worker alive
[AUTHOR] /authors/OL2690004A
Author worker alive
[WORKS] science -> 50

In [ ]:
import requests, threading, queue, time, json, logging
import boto3
import pandas as pd
import io

BASE = "https://openlibrary.org"
NUM_THREADS = 8
AUTHOR_THREADS = 4
SUBJECT_THREADS = 4
SLEEP = 0.1
RETRY = 3

bucket = "mhai-bk"

# ================= INIT =================
session = requests.Session()
s3 = boto3.client("s3")

work_queue = queue.Queue(maxsize=20000)
author_queue = queue.Queue(maxsize=20000)
subject_queue = queue.Queue()

lock = threading.Lock()

logging.basicConfig(level=logging.INFO, filename="crawler.log")

# ================= GLOBAL DEDUP =================
seen_editions = set()
seen_authors = set()

# ================= LOAD/SAVE SUBJECT =================
def load_seen_subjects():
    try:
        obj = s3.get_object(Bucket=bucket, Key="meta/subjects.txt")
        return set(obj["Body"].read().decode().splitlines())
    except:
        return set()

def save_seen_subjects(subjects):
    body = "\n".join(subjects)
    s3.put_object(Bucket=bucket, Key="meta/subjects.txt", Body=body)

# ================= CLEAN =================
def clean_dataframe(df):
    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
        )
    return df

# ================= SAVE PARQUET =================
def upload_parquet(records, prefix):
    if not records:
        return

    df = pd.DataFrame(records)
    df = clean_dataframe(df)

    buffer = io.BytesIO()
    df.to_parquet(buffer, engine="pyarrow", compression="snappy", index=False)
    buffer.seek(0)

    key = f"raw_data/{prefix}/part-{int(time.time()*1000)}.parquet"

    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=buffer.getvalue(),
        ContentType="application/octet-stream"
    )

# ================= FETCH =================
def fetch(url):
    for i in range(RETRY):
        try:
            r = session.get(url, timeout=10)
            if r.status_code == 200:
                return r.json()
        except Exception:
            logging.warning(f"Retry {i} failed: {url}")
            time.sleep(1)
    return None

# ================= GET ALL SUBJECT =================
def get_all_subjects():
    subjects = set()
    offset = 0

    while True:
        url = f"{BASE}/subjects.json?limit=1000&offset={offset}"
        data = fetch(url)

        if not data or not data.get("subjects"):
            break

        for s in data["subjects"]:
            name = s.get("key")
            if name:
                subjects.add(name)

        print(f"[SUBJECT] total: {len(subjects)}")

        offset += 1000
        time.sleep(SLEEP)

    return list(subjects)

# ================= CRAWL SUBJECT =================
def crawl_subject(subject):
    offset = 0
    seen_local = set()

    while True:
        url = f"{BASE}/subjects/{subject}.json?limit=50&offset={offset}"
        data = fetch(url)

        if not data or not data.get("works"):
            break

        records = []

        for w in data["works"]:
            wid = w["key"]

            if wid in seen_local:
                continue
            seen_local.add(wid)

            records.append(w)
            work_queue.put(wid)

            for a in w.get("authors", []):
                aid = a.get("author", {}).get("key")
                if aid:
                    author_queue.put(aid)

        if records:
            with lock:
                upload_parquet(records, "works")

        print(f"[WORKS] {subject} -> {len(records)}")

        offset += 50
        time.sleep(SLEEP)

# ================= SUBJECT WORKER =================
def subject_worker(seen_subjects):
    while True:
        subject = subject_queue.get()

        if subject in seen_subjects:
            subject_queue.task_done()
            continue

        seen_subjects.add(subject)
        save_seen_subjects(seen_subjects)

        crawl_subject(subject)

        subject_queue.task_done()

# ================= EDITIONS WORKER =================
def worker_editions():
    batch = []

    while True:
        try:
            work_id = work_queue.get(timeout=5)
        except:
            continue

        offset = 0

        while True:
            url = f"{BASE}{work_id}/editions.json?limit=100&offset={offset}"
            data = fetch(url)

            if not data or not data.get("entries"):
                break

            for e in data["entries"]:
                key = e.get("key")

                with lock:
                    if key and key not in seen_editions:
                        seen_editions.add(key)
                        batch.append(e)

                for a in e.get("authors", []):
                    aid = a.get("key")
                    if aid:
                        author_queue.put(aid)

            offset += 100

        # batch lớn mới ghi → tránh small file
        if len(batch) >= 2000:
            with lock:
                upload_parquet(batch, "editions")
                batch.clear()

        work_queue.task_done()
        time.sleep(SLEEP)

# ================= AUTHORS WORKER =================
def worker_authors():
    batch = []

    while True:
        try:
            aid = author_queue.get(timeout=5)
        except:
            continue

        if aid in seen_authors:
            author_queue.task_done()
            continue

        seen_authors.add(aid)

        data = fetch(f"{BASE}{aid}.json")

        if data:
            batch.append(data)

        if len(batch) >= 1000:
            with lock:
                upload_parquet(batch, "authors")
                batch.clear()

        author_queue.task_done()
        time.sleep(SLEEP)

# ================= MAIN =================
def main():
    print("Load subjects...")
    subjects = get_all_subjects()

    print(f"Total subjects: {len(subjects)}")

    seen_subjects = load_seen_subjects()

    # push vào queue
    for s in subjects:
        subject_queue.put(s)

    print("Start workers...")

    # subject workers
    for _ in range(SUBJECT_THREADS):
        threading.Thread(target=subject_worker, args=(seen_subjects,), daemon=True).start()

    # edition workers
    for _ in range(NUM_THREADS):
        threading.Thread(target=worker_editions, daemon=True).start()

    # author workers
    for _ in range(AUTHOR_THREADS):
        threading.Thread(target=worker_authors, daemon=True).start()

    subject_queue.join()
    work_queue.join()
    author_queue.join()

    print("DONE")

# ================= LOOP FOREVER =================
if __name__ == "__main__":
    while True:
        main()
        print("Sleep 1 hour...")
        time.sleep(3600)